<a href="https://colab.research.google.com/github/jkh02/-XAI-LLM-/blob/main/XAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
from google.colab import drive

drive.mount('/content/drive')

# 2. 내 드라이브 내 프로젝트 전용 폴더 생성 및 이동
# 'MyProject' 대신 본인이 원하는 폴더명을 적으세요.
project_path = "/content/drive/MyDrive/MyProject"

if not os.path.exists(project_path):
    os.makedirs(project_path)
    print(f" 새 폴더를 생성했습니다: {project_path}")

os.chdir(project_path)
print(f" 현재 작업 위치: {os.getcwd()}")

Mounted at /content/drive
 현재 작업 위치: /content/drive/MyDrive/MyProject


In [1]:
# 1. 1.24.4 대신 <2.0.0으로 설정하여 빌드 에러를 방지하고 spacy 호환성을 맞춥니다.
!pip install "numpy<2.0.0" spacy

# 아까 덜 깔린 부품들만 다시 깔아줍니다. (넘파이는 이미 해결됐으니 패스!)
!pip install -q transformers peft accelerate bitsandbytes captum langchain langchain-community pypdf faiss-cpu sentence-transformers datasets
print(" 이제  준비되었습니다!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
 이제  준비되었습니다!


In [5]:
import os
import json
import torch
from google.colab import drive
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from captum.attr import LayerIntegratedGradients
from datasets import Dataset
from datetime import datetime

# 경로 기본 세팅
project_path = "/content/drive/MyDrive/MyProject"
save_path = os.path.join(project_path, "saved_weights")

# ==========================================
# 1단계: JSONL 데이터 로드 및 지식 창고 생성
# ==========================================
print("임베딩 모델을 준비합니다...")
embeddings = HuggingFaceEmbeddings(model_name="snunlp/KR-SBERT-V40K-klueNLI-augSTS")

# 데이터 경로 (이중 폴더 주의)
jsonl_file_path = os.path.join(project_path, "data1", "chunks.jsonl")

docs = []
print("JSONL 파일에서 데이터를 읽어옵니다...")

try:
    with open(jsonl_file_path, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            doc = Document(
                page_content=item["chunk_text"],
                metadata={
                    "title": item.get("title", ""),
                    "doc_id": item.get("doc_id", ""),
                    "source": item.get("source_url", "")
                }
            )
            docs.append(doc)
    print(f"완료: 총 {len(docs)}개의 문서 청크를 불러왔습니다.")

except Exception as e:
    print(f"데이터 로드 실패: {e}")

if docs:
    print("지식 창고(FAISS)를 구축 중입니다...")
    vectorstore = FAISS.from_documents(documents=docs, embedding=embeddings)
    faiss_save_path = os.path.join(project_path, "saved_faiss_index_json")
    vectorstore.save_local(faiss_save_path)
    print("지식 창고 저장 완료")

# 🚨 안전장치: 데이터가 없으면 중단
if len(docs) == 0:
    raise RuntimeError("데이터를 찾을 수 없습니다. 경로를 확인해주세요.")

# ==========================================
# 2단계: AI 모델 및 XAI(Captum) 세팅
# ==========================================
print("2단계: Qwen 모델 및 가중치 세팅 중...")
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")

# 모델 구조가 다를 경우 기존 가중치를 불러오지 않도록 처리
if os.path.exists(os.path.join(save_path, "adapter_config.json")):
    print("기존 학습 가중치를 불러옵니다.")
    from peft import PeftModel
    model = PeftModel.from_pretrained(base_model, save_path, is_trainable=True)
else:
    print("새로운 LoRA 어댑터를 생성합니다.")
    # Qwen 모델에 최적화된 타겟 모듈 설정
    config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(base_model, config)

# ✅ Qwen 모델 구조에 맞게 Captum 타겟 레이어 경로 수정
target_layer = model.base_model.model.model.embed_tokens
lig = LayerIntegratedGradients(model, target_layer)

# ==========================================
# 2.5단계: 데이터 변환 및 모델 학습 (Fine-Tuning)
# ==========================================
print("학습용 데이터 변환 중...")
train_texts = [doc.page_content for doc in docs]
dataset = Dataset.from_dict({"text": train_texts})

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 체크포인트 경로 설정
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_dir = os.path.join(project_path, f"checkpoints_{timestamp}")

training_args = TrainingArguments(
    output_dir=checkpoint_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    save_strategy="epoch",
    logging_steps=10,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    data_collator=data_collator,
)

print("모델 파인튜닝 시작")
trainer.train()

# ==========================================
# 2.6단계: 최종 학습된 모델 저장
# ==========================================
if not os.path.exists(save_path):
    os.makedirs(save_path)
trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print("학습 및 저장 완료")

임베딩 모델을 준비합니다...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: snunlp/KR-SBERT-V40K-klueNLI-augSTS
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


JSONL 파일에서 데이터를 읽어옵니다...
완료: 총 100개의 문서 청크를 불러왔습니다.
지식 창고(FAISS)를 구축 중입니다...
지식 창고 저장 완료
2단계: Qwen 모델 및 가중치 세팅 중...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

새로운 LoRA 어댑터를 생성합니다.
학습용 데이터 변환 중...


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

모델 파인튜닝 시작


Step,Training Loss
10,2.531073
20,2.234024
30,2.115244


학습 및 저장 완료


In [1]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# ==========================================
# 1. 저장된 데이터와 모델 불러오기
# ==========================================
print("저장된 데이터와 모델을 불러오는 중입니다...")

project_path = "/content/drive/MyDrive/MyProject"
faiss_save_path = os.path.join(project_path, "saved_faiss_index_json")
save_path = os.path.join(project_path, "saved_weights")

# 1-1. FAISS 로드
embeddings = HuggingFaceEmbeddings(model_name="snunlp/KR-SBERT-V40K-klueNLI-augSTS")
vectorstore = FAISS.load_local(faiss_save_path, embeddings, allow_dangerous_deserialization=True)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 1-2. Qwen 모델 및 LoRA 가중치 로드
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Qwen용 패딩 토큰 설정
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base_model, save_path)

print("챗봇 준비 완료!\n")

# ==========================================
# 2. 대화형 질문 및 답변 생성
# ==========================================
print("\n==================================================")
print("복지 정책 챗봇이 준비되었습니다! (종료하려면 '종료' 또는 'q' 입력)")
print("==================================================")

while True:
    question = input("\n질문을 입력하세요: ")

    if question.lower() in ['종료', 'q', 'quit', 'exit']:
        print("종료합니다.")
        break

    if not question.strip():
        continue

    # 문서 검색
    docs = retriever.invoke(question)
    context_list = []
    for doc in docs:
        title = doc.metadata.get("title", "제목 없음")
        content = doc.page_content
        context_list.append(f"[정책명: {title}]\n내용: {content}")

    context = "\n\n".join(context_list)

    # ✅ Qwen 전용 ChatML 프롬프트 양식으로 변경
    prompt = f"<|im_start|>system\n당신은 대한민국의 친절한 복지 정책 안내 챗봇입니다. 주어진 문서의 내용만을 바탕으로 명확하고 자연스러운 한국어로 답변하세요.<|im_end|>\n<|im_start|>user\n다음 문서를 참고하여 질문에 답해주세요.\n\n[문서]\n{context}\n\n질문: {question}<|im_end|>\n<|im_start|>assistant\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    print("답변을 생성 중입니다...\n")

    # 모델 추론
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,      # 답변 길이를 위해 조금 늘렸습니다.
        do_sample=True,
        temperature=0.1,
        top_p=0.9,
        repetition_penalty=1.1,  # Qwen은 1.1 정도가 적당합니다.
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    # 답변 추출 로직
    input_length = inputs.input_ids.shape[1]
    generated_tokens = outputs[0][input_length:]
    final_answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    print("--------------------------------------------------")
    print(final_answer)
    print("--------------------------------------------------")

저장된 데이터와 모델을 불러오는 중입니다...


/tmp/ipykernel_6075/20616159.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="snunlp/KR-SBERT-V40K-klueNLI-augSTS")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: snunlp/KR-SBERT-V40K-klueNLI-augSTS
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

챗봇 준비 완료!


복지 정책 챗봇이 준비되었습니다! (종료하려면 '종료' 또는 'q' 입력)

질문을 입력하세요: 저소득층을 위한 정책 추천해줘.
답변을 생성 중입니다...

--------------------------------------------------
저소득층을 위한 다양한 정책이 있습니다. 먼저, 공공분양 정책을 통해 저소득층에게 주택을 제공하여 주거안정을 도모합니다. 또한, 양곡할인 정책을 통해 저소득층과 차상위계층에게 식량비를 절감하여 생활안정을 지원합니다. 이러한 정책들은 저소득층의 삶의 질 향상을 위해 중요한 역할을 합니다.
--------------------------------------------------

질문을 입력하세요: 청년들을 위한 정책 추천해줘
답변을 생성 중입니다...

--------------------------------------------------
청년들을 위한 정책 추천드립니다. 먼저 '햇살론 youth 보증사업'이 있습니다. 이는 서민금융 활성화를 통해 금융취약계층인 대학생과 청년에게 금융애로를 해소하여 학업 및 취업에 전념할 수 있도록 지원합니다. 또한 '여성새로일하기지원센터 사업'에서는 경력단절여성을 대상으로 취업상담, 직업교육, 인턴, 취업연계 및 사후관리 등 종합적인 취업서비스를 제공합니다. 이러한 정책들이 청년들의 창의적이고 윤택한 성장을 돕습니다.
--------------------------------------------------

질문을 입력하세요: q
종료합니다.
